---

# Dudley Disability Data Cleaning

---

## Purpose

Clean and validate Census 2021 disability data for Dudley residents aged
16 years and over, ready for later analysis of local employment barriers
and possible support needs.

The source is RM019 — Economic activity status by hours worked by
disability, accessed through ONS/Nomis. The economic activity selection
is Total, covering all economic activity statuses.

## Objectives

- Inspect the raw file structure and load the dataset.
- Identify and handle blank rows and other non-data rows.
- Check missing values, duplicates and LSOA identifiers.
- Standardise column names and validate numeric counts.
- Save a cleaned CSV and document the cleaning decisions.

## Inputs

- `raw_datasets/dudley_disability_16_plus.csv`
- Census 2021 data downloaded from Nomis on 16 October 2026.

## Outputs

- `cleaned_datasets/dudley_disability_16_cleaned.csv`
- Documented checks, findings and a cleaning conclusion.

## Ethical Considerations

Disability status does not establish a person's ability or willingness
to work, or their individual support needs. These neighbourhood statistics
must not be used to make assumptions about individual residents.

The population includes everyone aged 16 and over, including older
residents. It must not be described as exclusively working age.

## Additional Comments

The raw source file will remain unchanged during cleaning.
Percentage calculations, analysis, visualisations and modelling will
be completed in separate notebooks.

Code developed with assistance from ChatGPT will be acknowledged,
with relevant documentation linked in comments.

---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Dudley employment barriers'

### Import libraries

Import pandas to load, inspect and clean the disability dataset.

In [4]:
# Import pandas for loading and cleaning the dataset.
import pandas as pd

## 1. Inspect the raw file structure

Read the first ten lines to check the dataset title, population and
column-heading row before loading the CSV with pandas.

In [5]:
# Developed with assistance from ChatGPT.
raw_path = "raw_datasets/dudley_disability_16_plus.csv"

# Read the source headings without changing the raw file.
with open(raw_path, "r", encoding="utf-8-sig") as file:
    for line_number in range(1, 11):
        line = file.readline()
        if not line:
            break
        print(f"{line_number}: {line.rstrip()}")

1: RM019 - Economic activity status by hours worked by disability,,,,,,
2: ONS Crown Copyright Reserved [from Nomis on 16 October 2026],,,,,,
3: Population :,All usual residents aged 16 years and over,,,,,
4: Units      :,Persons,,,,,
5: Date       :,2021,,,,,
6: Economic activity status and hours worked:,Total,,,,,
7: ,,,,,,
8: 2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
9: ,,,,,,
10: Dudley 001A,E01009763,1640,350,154,196,1290


### Findings

- The file contains Census 2021 RM019 disability data, downloaded
  from Nomis on 16 October 2026.
- The population is all usual residents aged 16 years and over.
  Counts are measured in persons.
- The economic activity selection is Total, so the data includes
  all economic activity statuses.
- Column headings appear on line 8. Skip the first seven lines
  when loading the CSV.
- Line 9 is an empty record containing commas. This will be
  checked during cleaning.
- The table contains LSOA names, codes and five count columns.
- The two categories “limited a lot” and “limited a little” are
  subcategories of the disabled total. They must not be added
  to that total, as this would double-count residents.

---

## 2. Load and preview the dataset

Load the CSV, skipping the seven introductory lines so that line 8
provides the column headings. Preview five rows and check the initial
number of rows and columns before cleaning.

In [6]:
# Developed with assistance from ChatGPT.
# Load the data using the column headings identified during inspection.
disability_df = pd.read_csv(
    raw_path,
    skiprows=7,
    encoding="utf-8-sig"
)

# Check the initial dimensions and preview five rows.
print(f"Rows: {disability_df.shape[0]}")
print(f"Columns: {disability_df.shape[1]}")
disability_df.head()

Rows: 211
Columns: 7


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dudley 001A,E01009763,1640.0,350.0,154.0,196.0,1290.0
2,Dudley 001B,E01009764,1264.0,261.0,121.0,140.0,1003.0
3,Dudley 001C,E01009767,1268.0,296.0,145.0,151.0,972.0
4,Dudley 001D,E01009769,1428.0,376.0,208.0,168.0,1052.0


### Findings

- The initial dataset contains 211 rows and 7 columns.
- The column headings have loaded correctly.
- The first row contains missing values in every column, matching
  the empty record identified in the raw file.
- The following rows contain Dudley LSOA names, codes and counts.
- Counts currently display with decimal places, such as 1640.0.
  Numeric types will be checked during cleaning.
- The 211 rows are not yet a verified count of neighbourhoods.
  Blank rows and any source notes must be inspected before removal.

---

## 3. Inspect blank rows and rows without area codes

Count completely empty rows and inspect any rows without a mnemonic
(LSOA code). This helps identify blank records and source notes before
deciding which rows to remove.

In [7]:
# Developed with assistance from ChatGPT.

# Identify rows where every value is missing.
blank_rows = disability_df.isna().all(axis=1)

# Identify missing or whitespace-only area codes.
missing_codes = (
    disability_df["mnemonic"].isna()
    | disability_df["mnemonic"].astype(str).str.strip().eq("")
)

print(f"Completely empty rows: {blank_rows.sum()}")
print(f"Rows without an area code: {missing_codes.sum()}")

# Show these rows for inspection before removing anything.
disability_df.loc[missing_codes]

Completely empty rows: 4
Rows without an area code: 4


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
206,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Findings

- Four rows are completely empty: indices 0, 204, 205 and 206.
- These are also the only four rows without an area code.
- These blank records can be removed without losing population data.
- Removing them would leave 207 rows. These must still be checked
  to distinguish valid Dudley LSOAs from any other records.
- Source notes may contain text in the mnemonic column, so checking
  missing codes alone does not identify every possible non-data row.

---

## 4. Inspect other non-data records

Check for non-empty rows whose mnemonic does not match the format of
an English LSOA code: E followed by eight digits.

Inspect these records before deciding whether to remove them.
Matching the code format alone does not confirm Dudley coverage.

In [8]:
# Developed with assistance from ChatGPT.

# Check the format of each area code.
valid_code_format = (
    disability_df["mnemonic"]
    .astype("string")
    .str.fullmatch(r"E\d{8}", na=False)
)

# Identify non-empty rows without a correctly formatted code.
other_records = disability_df.loc[
    ~blank_rows & ~valid_code_format
]

print(f"Rows matching LSOA code format: {valid_code_format.sum()}")
print(f"Other non-empty records: {len(other_records)}")

# Display the records before making a cleaning decision.
other_records

Rows matching LSOA code format: 203
Other non-empty records: 4


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
207,NaN,In order to protect against disclosure of pers...,NaN,NaN,NaN,NaN,NaN
208,NaN,have been swapped between different geographic...,NaN,NaN,NaN,NaN,NaN
209,NaN,by small amounts. Small counts at the lowest g...,NaN,NaN,NaN,NaN,NaN
210,NaN,affected.,NaN,NaN,NaN,NaN,NaN


### Findings

- 203 rows have a mnemonic matching the expected LSOA code format.
- Four additional non-empty rows, at indices 207–210, contain
  disclosure-control notes rather than neighbourhood records.
- These notes appear in the mnemonic column and have no population
  counts. They can be excluded from the cleaned table while remaining
  preserved in the raw CSV.
- Together with the four blank rows, this accounts for all eight
  non-data records in the initial 211 rows.
- The remaining 203 records still require checks for duplicates,
  missing values and Dudley coverage.

---

## 5. Remove the identified non-data rows

Keep the records with correctly formatted LSOA codes, excluding the
four blank rows and four source-note rows identified above.

Create a separate cleaning dataframe and reset its row index.
The original dataframe and raw CSV remain unchanged.

In [9]:
# Developed with assistance from ChatGPT.

# Keep the area records identified during inspection.
disability_clean = (
    disability_df.loc[valid_code_format]
    .copy()
    .reset_index(drop=True)
)

# Confirm how many records were removed and retained.
print(f"Rows removed: {len(disability_df) - len(disability_clean)}")
print(f"Rows remaining: {len(disability_clean)}")
print(f"Columns: {disability_clean.shape[1]}")

# Preview the first five retained records.
disability_clean.head()

Rows removed: 8
Rows remaining: 203
Columns: 7


,2021 super output area - lower layer,mnemonic,Total,Disabled under the Equality Act,Disabled under the Equality Act: Day-to-day activities limited a lot,Disabled under the Equality Act: Day-to-day activities limited a little,Not disabled under the Equality Act
0,Dudley 001A,E01009763,1640.0,350.0,154.0,196.0,1290.0
1,Dudley 001B,E01009764,1264.0,261.0,121.0,140.0,1003.0
2,Dudley 001C,E01009767,1268.0,296.0,145.0,151.0,972.0
3,Dudley 001D,E01009769,1428.0,376.0,208.0,168.0,1052.0
4,Dudley 001E,E01009770,1213.0,278.0,136.0,142.0,935.0


### Findings

- Eight non-data rows were removed: four blank records and four
  disclosure-control note rows.
- The cleaning dataframe now contains 203 rows and 7 columns.
- The index has been reset, and the preview begins with Dudley 001A.
- All original columns and retained population counts are preserved.
- The raw CSV and original dataframe remain unchanged.
- Further validation is needed before the dataset is saved.

---

## 6. Standardise column names

Rename the columns to make them easier to use and interpret.
Use LSOA_Name and LSOA_Code consistently with the other cleaned datasets.

Include the 16-plus population in count column names. Keep the disabled
total separate from its two activity-limitation subcategories.

In [10]:
# Developed with assistance from ChatGPT.

# Map the original headings to clear, consistent names.
column_names = {
    "2021 super output area - lower layer": "LSOA_Name",
    "mnemonic": "LSOA_Code",
    "Total": "Total_16_Plus",
    "Disabled under the Equality Act": "Disabled_16_Plus",
    (
        "Disabled under the Equality Act: "
        "Day-to-day activities limited a lot"
    ): "Limited_A_Lot_16_Plus",
    (
        "Disabled under the Equality Act: "
        "Day-to-day activities limited a little"
    ): "Limited_A_Little_16_Plus",
    "Not disabled under the Equality Act": "Not_Disabled_16_Plus"
}

disability_clean = disability_clean.rename(columns=column_names)

# Check the updated headings and preview five rows.
print(disability_clean.columns.tolist())
disability_clean.head()

['LSOA_Name', 'LSOA_Code', 'Total_16_Plus', 'Disabled_16_Plus', 'Limited_A_Lot_16_Plus', 'Limited_A_Little_16_Plus', 'Not_Disabled_16_Plus']


,LSOA_Name,LSOA_Code,Total_16_Plus,Disabled_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Little_16_Plus,Not_Disabled_16_Plus
0,Dudley 001A,E01009763,1640.0,350.0,154.0,196.0,1290.0
1,Dudley 001B,E01009764,1264.0,261.0,121.0,140.0,1003.0
2,Dudley 001C,E01009767,1268.0,296.0,145.0,151.0,972.0
3,Dudley 001D,E01009769,1428.0,376.0,208.0,168.0,1052.0
4,Dudley 001E,E01009770,1213.0,278.0,136.0,142.0,935.0


### Findings

- All seven columns have been renamed successfully.
- LSOA_Name and LSOA_Code provide consistent identifiers for later
  matching with the other cleaned datasets.
- The five count columns explicitly identify the population as 16-plus.
- The disabled total and its two activity-limitation subcategories
  remain separate.
- Renaming changed only the headings; the data values are unchanged.

---

## 7. Check missing values and duplicates

Check each column for missing values, then check for duplicate rows
and repeated LSOA codes. Each neighbourhood should have one record.

In [11]:
# Developed with assistance from ChatGPT.

# Count missing values in each column.
print("Missing values by column:")
print(disability_clean.isna().sum())

# Check for repeated records and area codes.
print(
    "\nDuplicate rows:",
    disability_clean.duplicated().sum()
)
print(
    "Duplicate LSOA codes:",
    disability_clean["LSOA_Code"].duplicated().sum()
)
print(
    "Unique LSOA codes:",
    disability_clean["LSOA_Code"].nunique()
)

Missing values by column:
LSOA_Name                   0
LSOA_Code                   0
Total_16_Plus               0
Disabled_16_Plus            0
Limited_A_Lot_16_Plus       0
Limited_A_Little_16_Plus    0
Not_Disabled_16_Plus        0
dtype: int64

Duplicate rows: 0
Duplicate LSOA codes: 0
Unique LSOA codes: 203


### Findings

- No missing values were found in any of the seven columns.
- There are no duplicate rows or repeated LSOA codes.
- The dataset contains 203 unique LSOA codes, with one record per code.
- No missing-value treatment or duplicate removal is needed.
- Dudley coverage will be checked separately.

---

## 8. Check Dudley neighbourhood coverage

Check that all area names follow the Dudley LSOA naming pattern.
Compare the LSOA codes with the cleaned economic activity dataset
to confirm that both datasets cover the same neighbourhoods.

In [12]:
# Developed with assistance from ChatGPT.

# Load the cleaned economic activity dataset for comparison.
economic_df = pd.read_csv(
    "cleaned_datasets/dudley_economic_activity_cleaned.csv"
)

# Check that all names follow the Dudley LSOA naming pattern.
dudley_names = disability_clean["LSOA_Name"].str.fullmatch(
    r"Dudley \d{3}[A-Z]",
    na=False
)

# Compare the sets of neighbourhood codes.
disability_codes = set(disability_clean["LSOA_Code"])
economic_codes = set(economic_df["LSOA_Code"])

print(f"Names matching Dudley LSOA pattern: {dudley_names.sum()}")
print(f"Disability LSOA codes: {len(disability_codes)}")
print(f"Economic activity LSOA codes: {len(economic_codes)}")
print(
    "Only in disability:",
    sorted(disability_codes - economic_codes)
)
print(
    "Only in economic activity:",
    sorted(economic_codes - disability_codes)
)

Names matching Dudley LSOA pattern: 203
Disability LSOA codes: 203
Economic activity LSOA codes: 203
Only in disability: []
Only in economic activity: []


### Findings

- All 203 area names match the Dudley LSOA naming pattern.
- Both datasets contain 203 unique LSOA codes.
- No codes appear exclusively in either dataset.
- The disability dataset covers exactly the same neighbourhoods
  as the cleaned economic activity dataset.
- LSOA_Code can therefore be used for matching these datasets
  during the later merging stage.

---

## 9. Validate population counts

Check the five count columns for non-numeric values, missing values,
negative numbers and fractional values before converting them to integers.

Population counts should be non-negative whole numbers.

In [13]:
# Developed with assistance from ChatGPT.

# Select the five population count columns.
count_columns = [
    "Total_16_Plus",
    "Disabled_16_Plus",
    "Limited_A_Lot_16_Plus",
    "Limited_A_Little_16_Plus",
    "Not_Disabled_16_Plus"
]

# Check numeric conversion without changing the cleaning dataframe.
numeric_counts = disability_clean[count_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Summarise issues separately for each column.
count_checks = pd.DataFrame({
    "Missing_or_non_numeric": numeric_counts.isna().sum(),
    "Negative_values": numeric_counts.lt(0).sum(),
    "Fractional_values": (
        numeric_counts.notna() & numeric_counts.mod(1).ne(0)
    ).sum()
})

count_checks

,Missing_or_non_numeric,Negative_values,Fractional_values
Total_16_Plus,0,0,0
Disabled_16_Plus,0,0,0
Limited_A_Lot_16_Plus,0,0,0
Limited_A_Little_16_Plus,0,0,0
Not_Disabled_16_Plus,0,0,0


### Findings

- All five count columns contain valid numeric values.
- No missing, non-numeric, negative or fractional values were found.
- The counts can be converted to integers without rounding or
  changing their values.
- No count corrections are needed based on these checks.

---

## 10. Convert population counts to integers

Convert the five validated count columns to integer data types.
This removes unnecessary decimal places without changing the counts.

In [14]:
# Developed with assistance from ChatGPT.

# Convert the validated whole-number counts to integers.
disability_clean[count_columns] = (
    disability_clean[count_columns].astype("int64")
)

# Confirm the resulting data types and preview five rows.
print(disability_clean[count_columns].dtypes)
disability_clean.head()

Total_16_Plus               int64
Disabled_16_Plus            int64
Limited_A_Lot_16_Plus       int64
Limited_A_Little_16_Plus    int64
Not_Disabled_16_Plus        int64
dtype: object


,LSOA_Name,LSOA_Code,Total_16_Plus,Disabled_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Little_16_Plus,Not_Disabled_16_Plus
0,Dudley 001A,E01009763,1640,350,154,196,1290
1,Dudley 001B,E01009764,1264,261,121,140,1003
2,Dudley 001C,E01009767,1268,296,145,151,972
3,Dudley 001D,E01009769,1428,376,208,168,1052
4,Dudley 001E,E01009770,1213,278,136,142,935


### Findings

- All five population count columns now use the int64 data type.
- The preview displays whole numbers without decimal places.
- Conversion preserved the validated counts; no rounding was needed.
- LSOA names and codes were unchanged.

---

## 11. Check consistency between population counts

Check whether the two activity-limitation subcategories sum to the
disabled total, and whether disabled plus not disabled equals the
total population aged 16 and over.

Report any differences for inspection without changing source counts.

In [15]:
# Developed with assistance from ChatGPT.

# Compare disability subcategories with the disabled total.
disabled_difference = (
    disability_clean["Limited_A_Lot_16_Plus"]
    + disability_clean["Limited_A_Little_16_Plus"]
    - disability_clean["Disabled_16_Plus"]
)

# Compare the two disability groups with the population total.
total_difference = (
    disability_clean["Disabled_16_Plus"]
    + disability_clean["Not_Disabled_16_Plus"]
    - disability_clean["Total_16_Plus"]
)

print(
    "Rows where disability subcategories do not match:",
    disabled_difference.ne(0).sum()
)
print(
    "Rows where disability groups do not match the total:",
    total_difference.ne(0).sum()
)

# Preview any records with differences.
difference_rows = disability_clean.loc[
    disabled_difference.ne(0) | total_difference.ne(0)
]
difference_rows.head()

Rows where disability subcategories do not match: 0
Rows where disability groups do not match the total: 0


,LSOA_Name,LSOA_Code,Total_16_Plus,Disabled_16_Plus,Limited_A_Lot_16_Plus,Limited_A_Little_16_Plus,Not_Disabled_16_Plus


### Findings

- For all 203 neighbourhoods, the two activity-limitation
  subcategories sum exactly to the disabled total.
- Disabled plus not disabled equals the total population aged
  16 and over in every row.
- No inconsistencies were found, so the differences table is empty.
- No adjustments to the source counts are needed.

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
